# Differentially expressed genes in neighborhoods
This notebook will perform differential abundance analysis, using the approach generated by Andreas Fønss Møller in the hackathon

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_milo
# python -m ipykernel install --user --name scrna_cartography_milo --display-name "milo"

#### Load libraries

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt 

import random
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_ann.kneighbors.annoy import AnnoyTransformer 

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Set up

In [2]:
# Create directories
mi.create_directories(dir_path = str(here('data/milo/files')))
mi.create_directories(dir_path = str(here('data/milo/plots')))
mi.create_directories(dir_path = str(here('data/milo/objects')))

/work/islet_cartography_scrna/data/milo/files Directory already exists!
/work/islet_cartography_scrna/data/milo/plots Directory already exists!
/work/islet_cartography_scrna/data/milo/objects Directory already exists!


In [3]:
# Paths
base_dir = str(here('data/milo/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

#### Load

In [ ]:
mdata =  md.read(os.path.join(objects_dir,"mdata_666_annotation.h5mu")) # object

In [5]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

#### Get mean expression per neighborhood

In [ ]:
# Get mean counts per neighborhood and save
milo.add_nhood_expression(mdata)
mean_nhood_expr = pd.DataFrame(
    mdata["milo"].varm["expr"].toarray(),
    index=mdata["milo"].var['index_cell'],        # neighbourhood IDs
    columns=mdata["rna"].var_names        # gene names
)
mean_nhood_expr.transpose().to_csv(
            os.path.join(files_dir, f"mean_counts_per_nhood.csv"),
            index_label="gene_symbol")

# save
mean_nhood_expr.transpose().to_csv(
            os.path.join(files_dir, f"mean_counts_per_nhood.csv"),
            index_label="gene_symbol")

##### T2DM

In [6]:
res_t2d = pd.read_csv(os.path.join(files_dir, "t2d_vs_nd.csv"), index_col=0) # diff abundance results
res_t2d.reset_index(drop=True, inplace = True)
df = pd.read_csv(os.path.join(files_dir, "cells_in_neighborhood.csv"), index_col=0) # cells in neighborhoods
df.reset_index(drop=True, inplace = True)

In [7]:
# Combine results
df['index_cell'] = df['neighborhood']
df = df.merge(res_t2d, how = "left", on = "index_cell")

In [8]:
# Add direction of regulation
df.loc[(df["SpatialFDR"] <= 0.1) & (df["logFC"] > 0), 'direction'] = 'up'
df.loc[(df["SpatialFDR"] <= 0.1) & (df["logFC"] < 0), 'direction'] = 'down'
df.loc[df["direction"].isna(), 'direction'] = "no_change"
# Add a key for differential expression
df['diff_key'] = df['nhood_annotation'] + '_' + df['direction']
# Set index
df = df.set_index('cell')

#### Diffenretially expressed genes one vs the rest

In [ ]:
nhood_anno = "nhood_annotation"
cluster_key = "diff_key"
sample_key = "ic_id_platform_adjusted_sample"
prefix = "t2d_vs_nd"
min_cells = 20 # minimum number of cells 
min_rep = 8 # minimum number of replicates

for anno_id in df[nhood_anno].unique():
    print(f"Processing {anno_id}")

    # Subset df to only contain annotation - drop duplicated cells and set cell as index (for merge)
    df_sub = df.loc[df[nhood_anno] == anno_id].copy()

    # Remove duplicated cells
    df_sub = df_sub[[cluster_key]].loc[~df_sub.index.duplicated(keep="first")]

    # Get cluster ids to test
    cluster_ids = [
        x
        for x in df_sub[cluster_key].unique()
        if isinstance(x, str) and (x.endswith("_up") or x.endswith("_down"))
    ]

    if len(cluster_ids) == 0:
        print(f"No differentally abundance neighborshoods for {anno_id} skipping...")
        continue

    # Create clustering columns
    for cluster_id in cluster_ids:
        print(f"Differential expression for {cluster_id}")
        col_name = f"{cluster_id}_vs_other"
        df_sub[col_name] = df_sub[cluster_key].apply(
            lambda x: cluster_id if x == cluster_id else "other"
        )

        # subset ann data object to neighborhoods of interest
        subset = adata[adata.obs_names.isin(df_sub.index.unique())].copy()

        subset.obs = subset.obs.merge(
            df_sub, how="left", left_index=True, right_index=True
        )

        # Check that we have enough cells per samples and samples for deseq analysis 
        counts = (
            subset.obs
            .groupby([sample_key, col_name])
            .size()
            .unstack(fill_value=0)
        )
        
        valid_cluster = (counts.get(cluster_id, 0) >= min_cells).sum()
        valid_other = (counts.get("other", 0) >= min_cells).sum()
        
        if valid_cluster < min_rep or valid_other < min_rep:
            print(f"Skipping {cluster_id}: insufficient replicates")
            continue

        # Remove samples with too few cells
        valid_samples_cluster = counts.index[counts.get(cluster_id, 0) >= min_cells]
        valid_samples_other = counts.index[counts.get("other", 0) >= min_cells]
        valid_samples = set(valid_samples_cluster) | set(valid_samples_other)

        # Remove samples from adata object
        subset = subset[subset.obs[sample_key].isin(valid_samples)].copy()

        dds = dg.prepare_pseudobulk_deseq_analysis(
            ad=subset,
            sample_key=sample_key,
            cluster_key=col_name,
            no_subset=True,
            n_cells=25,
            design=f"~ {sample_key} + {col_name}",
            layer="counts",
            func="sum",
            return_all=False,
            workers=30,
        )

        # Define comparison
        c1 = cluster_id
        c2 = "other"

        # Wald test
        res = dg.diff_genes_two_clusters(
            dds_obj=dds, cluster_index=col_name, cluster_1=c1, cluster_2=c2, workers=30
        )
        # Save results
        res.to_csv(
            os.path.join(files_dir, f"{prefix}_deg_{cluster_id}_vs_other.csv"),
            index_label="gene_symbol",
        )

        # Save normalized counts
        norm_df = pd.DataFrame(
            dds.layers["normed_counts"], index=dds.obs_names, columns=dds.var_names
        )

        norm_df.transpose().to_csv(
            os.path.join(files_dir, f"{prefix}_norm_counts_{cluster_id}_vs_other.csv"),
            index_label="gene_symbol",
        )

Processing beta
Differential expression for beta_down


Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.54 seconds.



In [ ]:
#https://github.com/scverse/pertpy/issues/680
def find_nhood_groups(
    mdata, 
    SpatialFDR_threshold = 0.1,
    merge_discord = False,
    max_lfc_delta=None,
    overlap=1,
    subset_nhoods: None | str = None,
):
    """Get igraph graph from adjacency matrix. Adjusted from scanpy louvain clustering."""
    # get neighborhoods
    nhs = mdata["rna"].obsm["nhoods"].copy()
    # get connectivities
    nhood_adj = mdata["milo"].varp["nhood_connectivities"].copy()
    # results from spatiel fdr (you should change this to your dataframe)
    da_res = mdata["milo"].var.copy()
    is_da = np.asarray(da_res.SpatialFDR <= SpatialFDR_threshold)

    # This is if you want to do it for a specific nhood annotation
    if subset_nhoods is not None:
        print(da_res.shape)
        mask_da_res = (
            da_res
            .query(subset_nhoods)
            .copy()
        )
        mask = np.asarray([True if x in mask_da_res.index else False for x in da_res.index])
        da_res = da_res.loc[mask, :].copy()
        print(da_res.shape)
        # mask = da_res.index.values.astype(bool)
        print(mask.shape)
        nhs = nhs[:, mask].copy()
        print(mask.shape)
        print(nhood_adj.shape)
        nhood_adj = nhood_adj[:, mask]
        nhood_adj = nhood_adj[mask, :].copy()
        print(nhood_adj.shape)
        # is_da = np.asarray(da_res.SpatialFDR <= SpatialFDR_threshold)
        is_da = is_da[mask].copy()
        # print(is_da.shape)

    # da_res.loc[is_da, 'logFC'].values @ (da_res.loc[is_da, 'logFC']).T.values
    # print(da_res.loc[is_da, 'logFC'])

    if merge_discord is False:
        # discord_sign = np.sign(da_res.loc[is_da, 'logFC'].values @ (da_res.loc[is_da, 'logFC'])) < 0
        # Assuming da.res is a pandas DataFrame and is.da is a boolean array
        discord_sign = np.sign(da_res.loc[is_da, 'logFC'].values @ da_res.loc[is_da, 'logFC'].values.T) < 0
        # print(discord_sign.shape)
        # nhood_adj[is_da, is_da][discord_sign] <- 0
        # print(nhood_adj.shape)
        nhood_adj[(is_da, is_da)][discord_sign] = 0

    if overlap > 1:
        nhood_adj[nhood_adj < overlap] = 0

    if max_lfc_delta is not None:  
        lfc_diff = np.array([da_res['logFC'].values - x for x in da_res['logFC'].values])
        nhood_adj[np.abs(lfc_diff) > max_lfc_delta] = 0

    # binarise
    # nhood_adj = np.asarray((nhood_adj > 0) + 0)
    nhood_adj = (nhood_adj > 0).astype(int)

    g = sc._utils.get_igraph_from_adjacency(nhood_adj, directed = False)

    weights = None
    part = g.community_multilevel(weights=weights)

    groups = np.array(part.membership)
    groups = groups.astype(int).astype(str)

    if subset_nhoods is not None:
        mdata["milo"].var["nhood_groups"] = np.nan
        mdata["milo"].var.loc[mask, "nhood_groups"] = groups
        return None
    mdata["milo"].var["nhood_groups"] = groups